## Checkpoint 5C — AMPDS2_01 five-seed Kernel SHAP

**Instructions:** Copy the *single code cell* below into the **existing verified Google Colab runtime** used for Checkpoints 4C, 5A and 5B. Do not start a new runtime. This cell reads the frozen model and corrected scoring file, reconstructs sampled training backgrounds for seeds 42/123/456/789/2026, validates archived seed-42 hashes, generates exactly five Kernel SHAP explanations, and writes only new evidence after all checks pass. Do not run other cells or modify settings.

In [ ]:
# CHECKPOINT 5C — AMPDS2_01 five-seed Kernel SHAP
# Original corrected scoring-CSV order; only background seed varies.
# No model fitting, threshold changes, LIME, or historical overwrites.

from pathlib import Path
from itertools import combinations
import hashlib, importlib.metadata as im, json
import joblib
import numpy as np
import pandas as pd
import shap

required = ['ROOT', 'D5', 'FEATURES', 'COLUMNS',
            'selected', 'STABILITY_PROTOCOL', 'protocol_hash']
missing = [v for v in required if v not in globals()]
assert not missing, f'STOP: Missing {missing}'

for pkg, version in {'scikit-learn':'1.6.1', 'shap':'0.54.1',
                     'numpy':'2.4.6', 'pandas':'2.2.3'}.items():
    assert im.version(pkg) == version, f'STOP: {pkg} changed'

seeds = [42, 123, 456, 789, 2026]
p = STABILITY_PROTOCOL
canonical = json.dumps(p, sort_keys=True, separators=(',', ':'))
assert hashlib.sha256(canonical.encode()).hexdigest() == protocol_hash
assert protocol_hash == '849e9d7ef15cc3b135ac74854f491ae3c2113f39c3ef527ca789cde8e829334c'
assert p['seeds'] == seeds
assert p['feature_order'] == list(FEATURES)
assert COLUMNS == [f'{f}_z' for f in FEATURES]
assert len(FEATURES) == 23
sp = p['kernel_shap']
assert (sp['background_size'], sp['nsamples'], sp['coalition_rng_seed']) == (100, 300, 42)
assert sp['l1_reg'] == 'num_features(10)'
assert sp['training_order']['AMPds2'] == 'scoring_csv'

manifest = json.loads((Path(D5)/'manifest.json').read_text())
entry = manifest['model_and_background']['AMPds2']
old = entry['shap_background_verification']
assert old['training_order'] == 'scoring_csv'
assert old['status'] == 'NUMERICALLY_CONFIRMED'
model_path = Path(entry['model_path'])
score_path = Path(entry['scoring_file'])
assert model_path.is_file() and score_path.is_file()

def sha(data): return hashlib.sha256(data).hexdigest()

def hashfile(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for part in iter(lambda: f.read(1024*1024), b''):
            h.update(part)
    return h.hexdigest()

assert hashfile(model_path) == entry['model_sha256']
model_amp = joblib.load(model_path)
assert model_amp.n_features_in_ == 23
assert len(model_amp.estimators_) == 300
assert model_amp.max_samples_ == 256

def amp_score(X):
    return -model_amp.decision_function(np.asarray(X, dtype=np.float32))

case = selected.loc[selected['window_id'].eq('AMPDS2_01')]
assert len(case) == 1 and case.iloc[0]['dataset'] == 'AMPds2'
x_amp = case[COLUMNS].to_numpy(dtype=np.float32)
assert x_amp.shape == (1,23) and np.isfinite(x_amp).all()
score_amp = float(amp_score(x_amp)[0])
assert abs(score_amp - float(case.iloc[0]['anomaly_score'])) <= 1e-9
print('PASS: Protocol, AMPds2 model and target window')

# Recover sampled training rows across all seeds in one scoring-CSV pass.
N_TRAIN = 441483
positions_by_seed = {s: np.random.default_rng(s).choice(N_TRAIN, 100, replace=False)
                     for s in seeds}
needed = sorted(set(int(j) for a in positions_by_seed.values() for j in a))
sampled = {}
training_seen = 0
for chunk in pd.read_csv(score_path, usecols=['split','anomaly_score']+COLUMNS,
                         chunksize=25000, float_precision='round_trip', low_memory=False):
    train_chunk = chunk.loc[chunk['split'].astype(str).eq('train')]
    n = len(train_chunk)
    if not n: continue
    hits = [q for q in needed if training_seen <= q < training_seen+n]
    if hits:
        sliced = train_chunk.iloc[[q-training_seen for q in hits]]
        Xs = sliced[COLUMNS].to_numpy(dtype=np.float32)
        Ss = sliced['anomaly_score'].to_numpy(dtype=np.float64)
        for pos, vector, stored in zip(hits, Xs, Ss):
            sampled[pos] = (vector, float(stored))
    training_seen += n
assert training_seen == N_TRAIN and len(sampled) == len(needed)
print(f'PASS: {N_TRAIN:,} training rows, {len(needed)} unique sampled positions')

backgrounds = {}
for s in seeds:
    pos = positions_by_seed[s]
    bg = np.stack([sampled[int(i)][0] for i in pos]).astype(np.float32)
    stored_scores = np.array([sampled[int(i)][1] for i in pos])
    assert bg.shape == (100,23) and np.isfinite(bg).all()
    assert np.max(np.abs(amp_score(bg)-stored_scores)) <= 1e-9
    backgrounds[s] = bg

assert sha(backgrounds[42].tobytes(order='C')) == old['background_sha256']
assert sha(positions_by_seed[42].astype('<i8').tobytes()) == old['sample_positions_sha256']
assert abs(float(amp_score(backgrounds[42]).mean()) - float(old['expected_value'])) <= 1e-9
print('PASS: AMPds2 historical seed-42 background and 100 archived scores')

runs, vectors, tops = [], [], {}
state = np.random.get_state()
try:
    for s in seeds:
        bg = backgrounds[s]
        np.random.seed(42)  # Hold coalition RNG fixed across runs.
        explainer = shap.KernelExplainer(amp_score, bg)
        phi = np.asarray(explainer.shap_values(x_amp, nsamples=300,
                            l1_reg='num_features(10)'), dtype=np.float64).reshape(-1)
        base = float(np.asarray(explainer.expected_value).reshape(-1)[0])
        assert phi.shape == (23,) and np.isfinite(phi).all()
        assert abs(base - float(amp_score(bg).mean())) <= 1e-9
        residual = abs(base + float(phi.sum()) - score_amp)
        assert residual <= 1e-6, f'STOP: SHAP additivity failed at seed {s}'
        order = np.lexsort((np.arange(23), -np.abs(phi)))
        top = order[:5].tolist()
        tops[s] = set(top)
        runs.append(dict(dataset='AMPds2',window_id='AMPDS2_01', seed=s,
            score=score_amp,expected_value=base,additivity_residual=residual,
            background_sha256=sha(bg.tobytes(order='C')),
            positions_sha256=sha(positions_by_seed[s].astype('<i8').tobytes()),
            nonzero_contributions=int(np.count_nonzero(phi)),
            top5_features='|'.join(FEATURES[i] for i in top)))
        for i, feature in enumerate(FEATURES):
            vectors.append(dict(dataset='AMPds2',window_id='AMPDS2_01',seed=s,
                feature_index=i,feature=feature,scaled_feature_value=float(x_amp[0,i]),
                shap_value=float(phi[i]),abs_shap_value=float(abs(phi[i])),
                top5=i in tops[s]))
        print(f'PASS seed={s}: additivity={residual:.3e}; top5={[FEATURES[i] for i in top]}')
finally:
    np.random.set_state(state)

assert len(runs) == 5 and len(vectors) == 115
pairs=[]
for a,b in combinations(seeds,2):
    A,B=tops[a],tops[b]
    pairs.append(dict(dataset='AMPds2',window_id='AMPDS2_01',seed_a=a,seed_b=b,
        intersection_count=len(A&B),union_count=len(A|B),top5_jaccard=len(A&B)/len(A|B)))
assert len(pairs)==10
j=np.array([v['top5_jaccard'] for v in pairs])
summary=dict(dataset='AMPds2',window_id='AMPDS2_01',protocol_sha256=protocol_hash,
    method='Kernel SHAP',vary='background_sampling_seed',coalition_rng_seed=42,
    historical_attribution_identity='NOT_ESTABLISHED',seeds=seeds,n_runs=5,n_seed_pairs=10,
    mean_pairwise_top5_jaccard=float(j.mean()),
    min_pairwise_top5_jaccard=float(j.min()),max_pairwise_top5_jaccard=float(j.max()),
    max_additivity_residual=float(max(row['additivity_residual'] for row in runs)))

# Only create new evidence after all five runs pass.
parent = Path(ROOT)/'SUPERVISOR_FINAL_EXPERIMENTS_2026'/'06_explanation_evaluation'
assert parent.is_dir()
output = parent/'CHECKPOINT5C_AMPDS201_SHAP_BACKGROUND_5SEEDS_849e9d7e'
def csvbytes(records):
    return pd.DataFrame(records).to_csv(index=False,float_format='%.17g').encode('utf-8')
payloads = {
    'ampds201_shap_runs.csv': csvbytes(runs),
    'ampds201_shap_23_features_by_seed.csv': csvbytes(vectors),
    'ampds201_shap_pairwise_jaccard.csv': csvbytes(pairs),
    'ampds201_shap_summary.json': (json.dumps(summary,indent=2,sort_keys=True,allow_nan=False)+'\n').encode('utf-8')
}
for name, data in payloads.items():
    path = output/name
    if path.exists(): assert path.read_bytes() == data, f'STOP: Existing evidence differs {path}'
output.mkdir(exist_ok=True)
for name, data in payloads.items():
    path = output/name
    if not path.exists():
        with path.open('xb') as f: f.write(data)
    assert path.read_bytes()==data

print('\n=== CHECKPOINT 5C SUMMARY ===')
print(pd.DataFrame(runs)[['seed','expected_value','additivity_residual',
                          'nonzero_contributions','top5_features']].to_string(index=False))
print('Mean pairwise top-5 Jaccard:', summary['mean_pairwise_top5_jaccard'])
print('Minimum Jaccard:', summary['min_pairwise_top5_jaccard'])
print('Maximum Jaccard:', summary['max_pairwise_top5_jaccard'])
print('Maximum additivity residual:', summary['max_additivity_residual'])
print('Evidence directory:', output)
print('PASS: AMPDS2_01_SHAP_FIVE_SEED_EXPERIMENT')
print('No detector retraining, no LIME, no historical overwrites.')
